In [1]:
import numpy as np

In [33]:
# Activation functions

def relu(x):
    return np.maximum(0,x)

def sigmoid(x):
    return 1/(1+np.exp(-x))

def erelu(x, alpha = 0.01):
    return np.where(x > 0, x, alpha * np.exp(x) - 1)
def erelu_grad(x, alpha = 0.01):
    fx = erelu(x)
    return np.where(x > 0, 1, fx + alpha)

# 2 features (X) with 5 samples (m)
# NN with 3 hidden layers (4, 3, 1(output layer))

In [3]:
np.random.seed(123)

X = np.random.randn(20, 100) # (features, samples)
y = (np.random.randn(1, 100) > 0)

hidden_nodes = []
h1_nodes = 32 # number of nodes in hidden layer 1
h2_nodes = 16 # number of nodes in hidden layer 2
h3_nodes = 8
h4_nodes = 4
hidden_nodes.append(h1_nodes) 
hidden_nodes.append(h2_nodes)
hidden_nodes.append(h3_nodes)
hidden_nodes.append(h4_nodes)

In [4]:
def layer_sizes(X, y, hidden_nodes):
    n_x0 = X.shape[0] # number of features in X
    n_x1 = hidden_nodes[0]
    n_x2 = hidden_nodes[1]
    n_x3 = hidden_nodes[2]
    n_x4 = hidden_nodes[3]
    n_y = y.shape[0]

    return n_x0, n_x1, n_x2, n_x3, n_x4,n_y

In [12]:
def int_params(n_x0, n_x1, n_x2, n_x3, n_x4,n_y):
    w1 = (np.random.randn(n_x1, n_x0)) 
    w2 = (np.random.randn(n_x2, n_x1)) 
    w3 = (np.random.randn(n_x3, n_x2))
    w4 = (np.random.randn(n_x4, n_x3))
    w5 = (np.random.randn(n_y, n_x4)) 
    # trick for w (number of nodes in current layer, number of nodes in prev layer)
    # for first w just take (number of nodes in current layer, number of features)

    b1 = np.random.randn(n_x1, 1)
    b2 = np.random.randn(n_x2, 1)
    b3 = np.random.randn(n_x3, 1)
    b4 = np.random.randn(n_x4, 1)
    b5 = np.random.randn(n_y, 1)
    
    # trick for w (number of nodes in current layer, 1)

    params = {"w1": w1, "b1": b1, "w2": w2, "b2": b2,  "w3": w3, "b3": b3, "w4": w4, "w5" : w5, "b4" : b4, "b5" : b5}
    return params

In [27]:
def fwp(X, params):
    w1 = params["w1"]
    b1 = params["b1"]
    w2 = params["w2"]
    b2 = params["b2"]
    w3 = params["w3"]
    b3 = params["b3"]
    w4 = params["w4"]
    b4 = params["b4"]
    w5 = params["w5"]
    b5 = params["b5"]

    # layer 1
    z1 = w1 @ X + b1
    a1 = np.tanh(z1)

    z2 = w2 @ a1 + b2
    a2 = relu(z2)

    z3 = w3 @ a2 + b3
    a3 = erelu(z3)

    z4 = w4 @ a3 + b4
    a4 = relu(z4)

    z5 = w5 @ a4 + b5
    a5 = sigmoid(z5)
    # a3 = np.clip(a3, 1e-15, 1 - 1e-15) #optional

    cache = { "z1": z1, "a1": a1, "z2": z2, "a2": a2,"z3": z3, "a3": a3, "a4" : a4, "z4" : z4, "a5" : a5, "z5" : z5}
    return a5, cache

In [30]:
def compute_cost(y, a5):
    m = y.shape[1] # total number of samples
    cost1 = np.sum(y * np.log(a5) + (1 - y) * np.log(1 - a5))
    cost = -cost1 / m
    cost = float(np.squeeze(cost))
    return cost

In [34]:
def bwp(X, y, params, cache):
    w1 = params["w1"]
    b1 = params["b1"]
    w2 = params["w2"]
    b2 = params["b2"]
    w3 = params["w3"]
    b3 = params["b3"]
    w4 = params["w4"]
    b4 = params["b4"]
    w5 = params["w5"]
    b5 = params["b5"]

    z1 = cache["z1"]
    a1 = cache["a1"]
    z2 = cache["z2"]
    a2 = cache["a2"]
    z3 = cache["z3"]
    a3 = cache["a3"]
    z4 = cache["z4"]
    a4 = cache["a4"]
    z5 = cache["z5"]
    a5 = cache["a5"]

    m = X.shape[1]

    dz5 = a5 - y
    dw5 = (dz5 @ a4.T) / m
    db5 = np.sum(dz5, axis=1, keepdims=True) / m
    
    da4 = w5.T @ dz5
    dz4 = da4 * (z4 > 0)
    dw4 = (dz4 @ a3.T) / m
    db4 = np.sum(dz4, axis=1, keepdims=True) / m
    
    da3 = w4.T @ dz4
    dz3 = da3 * (z3 > 0)
    dw3 = (dz3 @ a2.T) / m
    db3 = np.sum(dz3, axis=1, keepdims=True) / m
    
    da2 = w3.T @ dz3
    dz2 = da2 * (z2 > 0)
    dw2 = (dz2 @ a1.T) / m
    db2 = np.sum(dz2, axis=1, keepdims=True) / m
    
    da1 = w2.T @ dz2
    dz1 = da1 * (z1 > 0)
    dw1 = (dz1 @ X.T) / m
    db1 = np.sum(dz1, axis=1, keepdims=True) / m

    grades = {
        "dw1": dw1, "db1": db1,
        "dw2": dw2, "db2": db2,
        "dw3": dw3, "db3": db3,
        "dw4": dw4, "db4": db4,
        "dw5": dw5, "db5": db5
    }

    return grades

In [24]:
def update(params, grades, lr=0.01):
    W1 = params['w1']
    w2 = params['w2']
    w3 = params['w3']
    w4 = params["w4"]
    w5 = params["w5"]

    b1 = params['b1']
    b2 = params['b2']
    b3 = params['b3']
    b4 = params["b4"]
    b5 = params["b5"]


    dw1 = grades["dw1"]
    db1 = grades["db1"]
    dw2 = grades["dw2"]
    db2 = grades["db2"]
    dw3 = grades["dw3"]
    db3 = grades["db3"]
    dw4 = grades["dw4"]
    db4 = grades["db4"]
    dw5 = grades["dw5"]
    db5 = grades["db5"]
 
 
    W1 = W1 - lr*dw1
    b1 = b1 - lr*db1
    w2 = w2 - lr*dw2
    b2 = b2 - lr*db2
    w3 = w3 - lr*dw3
    b3 = b3 - lr*db3
    w4 = w4 - lr*dw4
    b4 = b4 - lr*db4
    w5 = w5 - lr*dw5
    b5 = b5 - lr*db5

    params = {'w1':W1,   
              "b1": b1,
            "w2": w2,
            "b2": b2,
            "w3": w3,
            "b3": b3,
            "w4" : w4,
            "b4" : b4,
            "w5" : w5,
            "b5" : b5
             }
    
    return params

In [35]:
def NN(X,y,hidden_nodes,itr=10000,print_cost=False):
    np.random.seed(123)
    n_x0, n_x1, n_x2, n_x3, n_x4,n_y = layer_sizes(X,y,hidden_nodes)
    params = int_params(n_x0, n_x1, n_x2, n_x3, n_x4,n_y)
    for i in range(0, itr):
        a5,cache = fwp(X,params)
        cost = compute_cost(y, a5)
        grades = bwp(X, y, params, cache)
        params = update(params, grades, lr= 0.01)

        if print_cost and i % 10000 == 0:
            print("cost %i: %f" % (i, cost))
    return params

In [36]:
NN(X,y,hidden_nodes,itr=100000,print_cost=True)

cost 0: 2.124719
cost 10000: 0.167797
cost 20000: 0.111026
cost 30000: 0.038599
cost 40000: 0.005432
cost 50000: 0.003152
cost 60000: 0.040643
cost 70000: 0.000776
cost 80000: 0.063301
cost 90000: 0.000249


{'w1': array([[ 2.04670621e+03, -2.67819511e+02, -1.03449133e+04,
          4.03593751e+03,  7.74883439e+03,  7.88521141e+03,
         -4.31348252e+03, -1.62059867e+03, -8.90009583e+02,
         -7.08018652e+03, -2.64268358e+03, -2.55062728e+03,
         -2.93546526e+03,  8.41951691e+02,  7.08746907e+02,
         -1.14370952e+04, -4.28306958e+03,  1.82089105e+03,
          4.80428875e+03,  5.84884327e+03],
        [-8.35420015e+03, -1.05049531e+04, -1.03043530e+04,
          3.81024359e+02,  5.67360699e+03, -4.83667112e+03,
          9.22073549e+03, -3.30941673e+03,  2.78707445e+03,
         -6.69999400e+01,  8.39873848e+03, -8.51455892e+03,
          7.49885308e+03, -4.85064118e+03, -6.80697082e+03,
         -7.44324009e+02,  4.25444665e+03,  5.58443881e+03,
         -1.92962973e+02, -2.59558978e+03],
        [ 9.79890077e+03,  9.23118322e+03,  5.72022056e+03,
          9.84453075e+02, -1.67586859e+04,  1.76896876e+03,
          1.03733247e+04,  2.65750234e+03, -1.43806593e+02,
      